In [ ]:
from pilot.models        import DeepONet, build_deeponet
from pilot.data          import ODEIterableDataset, LatinHypercubeSampler
from pilot.physics       import harm_osc
from pilot.training      import Trainer, build_dataloaders

import numpy as np
import matplotlib.pyplot as plt 
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Initialize Harmonic Oscillator Object 
k = 2.0
c = 0.1
system = harm_osc([k, c])

# Initialize Sampler Object 
sampler = LatinHypercubeSampler(

    dimensions=2,
    lows      = [-1.0, -1.0],
    highs     = [1.0, 1.0]
    
)

In [4]:
# Initialize Training and validation dataset

train_dataset    = ODEIterableDataset(size = 1000,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 10),
                                      output_mask  = None)

val_dataset      = ODEIterableDataset(size = 10,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 10),
                                      output_mask  = None)

In [5]:
# Set Up DeepONet configuration 

DEEPONET_CONFIG = {
    
    "hidden_size" : 128,
    "depth"       : 6,
    "latent_size" : 50,
    "input_size_b": 2,
    "input_size_t": 1,
    "output_size" : 2,
    "activation"  : "tanh",

}

# Initialize DeepONet network 

deeponet = build_deeponet(DEEPONET_CONFIG)

In [7]:
# Set Up training Configuration as well as optimizer and loss functions

TRAIN_CONFIG = {

    "num_epochs"     : 80,
    "learning_rate"  : 0.004049755912601296,
    "batch_size"     : 64,
    "num_workers"    : 2,
    "Save_model"     : True,
    "Save_directory" : "../../weights/best_2d_osc.pth"
}

optimizer = torch.optim.Adam
loss_fn   = torch.nn.MSELoss()


In [8]:
# Initialize Train Object and Train 

trainer   = Trainer(
    model         = deeponet,
    train_dataset = train_dataset, 
    val_dataset   = val_dataset, 
    optimizer     = optimizer,
    loss_fn       = loss_fn, 
    train_cfg     = TRAIN_CONFIG,
    model_cfg     = DEEPONET_CONFIG
)

trainer.run()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/enricp/.netrc.
wandb: Currently logged in as: nikpursals to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training: 100%|██████████| 80/80 [02:43<00:00,  2.04s/it]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
